### 0. Установка библиотек

In [ ]:
# !pip install langchain langchain-qdrant qdrant-client sentence-transformers langchain-text-splitters langchain-huggingface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.9/389.9 kB 12.1 MB/s eta 0:00:00


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

Mounted at /content/drive


### 1. Импорты

In [42]:
import os
import shutil
from google.colab import userdata
from pathlib import Path
from uuid import uuid4
from tqdm import tqdm

from langchain_text_splitters import RecursiveCharacterTextSplitter, MarkdownHeaderTextSplitter
from langchain_qdrant import QdrantVectorStore
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams
from qdrant_client.http.exceptions import UnexpectedResponse

In [7]:
# Безопасно получаем данные из Colab Secrets
QDRANT_URL = userdata.get('QDRANT_URL')
QDRANT_API_KEY = userdata.get('QDRANT_API_KEY')

In [8]:
# --- CONFIG ---
DOC_NAME = "sp_60_cleaned_v3.2.md"
VERSION = "3.2"
BASE_DIR = Path("/content/drive/MyDrive/Colab_Notebooks/rag_docs/data/extracted")

doc_path = str(BASE_DIR / DOC_NAME)


### 2. Чанкинг

In [9]:
# Читаем документ
with open(doc_path, "r", encoding="utf-8") as f:
    md_text = f.read()

#### 2.1 RecursiveCharacterTextSplitter

In [10]:
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,          # символы (≈300–400 токенов)
    chunk_overlap=240,        # 20% от chunk_size
    length_function=len,
    separators=[
        "\n\n",               # двойной перенос — граница раздела/абзаца
        "\n",                 # одиночный перенос — внутри абзаца, но лучше не резать
        ". ",                 # граница предложения
        ", ",                 # запятая
        " ",                  # слово
        ""                    # символ
    ]
)
recursive_chunks = recursive_splitter.split_text(md_text)

In [11]:
recursive_chunks[100]

'7.2.16  Системы  воздушного  душирования  для  подачи  воздуха  на  рабочие  места должны быть отдельными от систем другого назначения.\n\n7.2.17  Системы механической общеобменной вентиляции следует предусматривать для помещений складов категорий А, Б и В1 - В4 с выделениями горючих газов и паров. Для  помещений  складов  категорий  А  и  Б  вместимостью  более  10  т  необходимо предусматривать резервную систему механической вытяжной вентиляции на требуемый воздухообмен, размещая местное управление системами при входе.\n\nДопускается предусматривать удаление воздуха только из верхней зоны системами с естественным побуждением, если в указанных помещениях выделяемые газы и пары легче воздуха и требуемый воздухообмен не превышает двукратного в 1 ч.\n\n7.2.18 Системы механической общеобменной вытяжной вентиляции следует предусматривать для помещений складов с выделением вредных газов и паров, предусматривая  резервную  систему  механической  вытяжной  вентиляции  на  требуемый воздухооб

#### 2.2 MarkdownHeaderTextSplitter

In [12]:
md_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[("#", "section"), ("##", "subsection")]
)

In [13]:
# тестирование
test_md_text = """
# 6 Внутренние системы теплоснабжения и отопления
## 6.1 Системы теплоснабжения
Текст раздела 6.1...
# 7 Вентиляция, кондиционирование воздуха и воздушное отопление
## 7.1 Общие положения
Текст раздела 7.1...
"""
chunks = md_splitter.split_text(test_md_text)
for chunk in chunks:
    print(chunk.metadata)
    print(chunk.page_content[:100])

{'section': '6 Внутренние системы теплоснабжения и отопления', 'subsection': '6.1 Системы теплоснабжения'}
Текст раздела 6.1...
{'section': '7 Вентиляция, кондиционирование воздуха и воздушное отопление', 'subsection': '7.1 Общие положения'}
Текст раздела 7.1...


In [14]:
md_chunks = md_splitter.split_text(md_text)

In [15]:
md_chunks[11].__dict__

{'id': None,
 'metadata': {'subsection': '6.3 Трубопроводы'},
 'page_content': '6.3.1 Трубопроводы систем внутреннего теплоснабжения следует предусматривать из стальных, медных, латунных, термостойких полимерных (в том числе металлополимерных) труб.  \nНе следует в одном контуре использовать элементы системы, выполненные из меди и алюминиевых сплавов.  \nНе допускается использование бывших в употреблении и восстановленных стальных труб, материалов и арматуры в проектной документации на строительство, реконструкцию и  капитальный  ремонт  зданий  и  сооружений  повышенного  и  нормального  уровней ответственности.',
 'type': 'Document'}

In [16]:
chunk_num = 8
print(f"№ чанка: {chunk_num}")
print(f"Metadata: {md_chunks[chunk_num].metadata}")
print(f"Длина чанка: {len(md_chunks[chunk_num].page_content)}")
print(f"Текст: \n {md_chunks[chunk_num].page_content}")

№ чанка: 8
Metadata: {'subsection': '5. Расчетные параметры внутреннего и наружного воздуха'}
Длина чанка: 12639
Текст: 
 5.1  Параметры  микроклимата  помещений  (кроме  помещений,  для  которых  они установлены другими нормативными документами) следует принимать по  ГОСТ 30494, ГОСТ 12.1.005, и СанПиН 2.2.4.548 для обеспечения температуры воздуха, результирующей температуры помещения, относительной влажности воздуха и скорости движения воздуха в пределах указанных параметров в обслуживаемой или рабочей зонах помещений (на постоянных и непостоянных рабочих местах):  
а) в холодный период года в обслуживаемой зоне жилых помещений -температуру воздуха по оптимальным параметрам ГОСТ 30494;  
б) в холодный период года в обслуживаемой зоне общественных и административнобытовых зданий или в рабочей зоне производственных помещений - температуру воздуха минимальную из допустимых температур при отсутствии избытков теплоты в помещениях или в пределах  допустимых параметров в помещениях  с избыт

#### 2.3 Custom chunker

In [ ]:
def smart_chunking(md_text, max_chunk_size=1200):
    """Комбинированный подход.
    1. Текст разбивается по заголовкам (каждый раздел - чанк)
    2. Если раздел длинный — рекурсивное разбиение с сохранением заголовка
    3. h1 наследуется от предыдущей секции, если пустой"""

    # 1. Разбиение по заголовкам (уровни # и ##)
    headers_to_split_on = [("#", "h1"), ("##", "h2")]
    header_splitter = MarkdownHeaderTextSplitter(headers_to_split_on)
    sections = header_splitter.split_text(md_text)

    # 2. Дополнительное рекурсивное разбиение длинных секций
    recursive_splitter = RecursiveCharacterTextSplitter(
        chunk_size=max_chunk_size,
        chunk_overlap=int(max_chunk_size * 0.2),
        separators=["\n\n", "\n", ". ", " ", ""]
    )

    final_chunks = []
    last_h1 = ""
    for sec in sections:
        # Наследуем h1 от предыдущей секции, если пустой
        current_h1 = sec.metadata.get("h1", "") or last_h1
        if sec.metadata.get("h1", ""):
            last_h1 = current_h1

        # Если секция короткая, добавляем как есть
        if len(sec.page_content) <= max_chunk_size:
            final_chunks.append({
                "text": sec.page_content,
                "metadata": {
                    "doc_source": DOC_NAME,
                    "doc_id": "СП 60.13330.2020",
                    "doc_title": "Отопление, вентиляция и кондиционирование воздуха",
                    "h1": current_h1,
                    "h2": sec.metadata.get("h2", ""),
                    "chunk_type": "text"
                }
            })
        else:
            # Разбиваем длинную секцию
            sub_chunks = recursive_splitter.split_text(sec.page_content)
            for sub in sub_chunks:
                final_chunks.append({
                    "text": sub,
                    "metadata": {
                        "doc_source": DOC_NAME,
                        "doc_id": "СП 60.13330.2020",
                        "doc_title": "Отопление, вентиляция и кондиционирование воздуха",
                        "h1": current_h1,
                        "h2": sec.metadata.get("h2", ""),
                        "chunk_type": "text"
                    }
                })
    print(f"Создано {len(final_chunks)} чанков")
    return final_chunks

In [23]:
smart_chunks = smart_chunking(md_text)

Создано 398 чанков


In [29]:
smart_chunks[55]

{'text': '6.2.10 Потери давления в системах водяного отопления должны составлять:  \n-  в  стояках  однотрубных  систем  и приборных  узлах  вертикальных  двухтрубных систем -не менее 70 % общих потерь давления в циркуляционных кольцах без учета потерь давления в общих участках;  \n- в стояках однотрубных систем отопления с нижней разводкой подающей и верхней разводкой обратной магистрали -не менее 300 Па на каждый метр высоты стояка;  \n- в двухтрубных вертикальных и однотрубных горизонтальных системах отопления в\nциркуляционных  кольцах  через  верхние  приборы  (ветви) -не  менее  естественного давления в них при расчетных параметрах теплоносителя.  \nРасполагаемую разность давления воды в подающем и обратном трубопроводах для циркуляции воды в системе отопления следует определять с учетом давления, возникающего при охлаждении воды в трубах и отопительных приборах.  \nНеучтенные потери циркуляционного давления в системе отопления следует принимать равными 10 % максимальных потерь д

### 3. Эмбеддинги & Qdrant

#### 3.1 Загрузка моделей и подключение

In [32]:
# Загружаем модель эмбеддингов
model_name = "intfloat/multilingual-e5-large"
model_kwargs = {'device': 'cuda'}
encode_kwargs = {'normalize_embeddings': True, 'batch_size':128}

embeddings_model = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs,
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-large
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# Qdrant
# Подключение к Qdrant Cloud
client = QdrantClient(
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
)

# Параметры коллекций
vector_size = 1024  # размерность e5-large
collection_config = VectorParams(size=vector_size, distance=Distance.COSINE)

# Создание коллекций с проверкой на существование
def create_collection_if_not_exists(client, name, vectors_config):
    try:
        client.create_collection(name, vectors_config=vectors_config)
        print(f"Коллекция {name} создана")
    except UnexpectedResponse as e:
        if "already exists" in str(e):
            print(f"Коллекция {name} уже существует, пропускаем")
        else:
            raise e

create_collection_if_not_exists(client, "sp60_recursive", collection_config)
create_collection_if_not_exists(client, "sp60_smart", collection_config)

#### 3.2 Создание документов и загрузка в БД

#### 3.2.1 Для рекурсивного чанкинга

In [ ]:
# Создаем документы для Qdrant с UUID
recursive_docs = []
for i, chunk in enumerate(recursive_chunks):
    recursive_docs.append(Document(
        page_content=chunk,
        metadata={
            "source": DOC_NAME,
            "strategy": "recursive",
            "chunk_id": i
        }
    ))

recursive_uuids = [str(uuid4()) for _ in range(len(recursive_docs))]
print(f"✅ Создано {len(recursive_uuids)} рекурсивных документов с UUID")

In [45]:
# Индексация в Qdrant
# Рекурсивная коллекция
vector_store_recursive = QdrantVectorStore(
    embedding=embeddings_model,
    client=client,
    collection_name="sp60_recursive",
)

#загрузка документов батчами
batch_size = 256
total_batches = (len(recursive_docs) + batch_size - 1) // batch_size

with tqdm(total=total_batches, desc="Добавление батчей в Qdrant") as pbar:
    for i in range(0, len(recursive_docs), batch_size):
        batch_docs = recursive_docs[i:i+batch_size]
        batch_ids = recursive_uuids[i:i+batch_size]

        vector_store_recursive.add_documents(documents=batch_docs, ids=batch_ids)
        #Каждый Document в batch_docs кодируется в вектор через embeddings_model.
        #Сохраняется в коллекцию "sp60_recursive".
        #В качестве ID используется заранее сгенерированный uuid.
        pbar.update(1)

print(f"✅ Рекурсивные чанки ({len(recursive_docs)}) проиндексированы и добавлены в Qdrant")

Добавление батчей в Qdrant: 100%|██████████| 2/2 [00:33<00:00, 16.55s/it]

✅ Рекурсивные чанки (401) проиндексированы и добавлены в Qdrant


#### 3.2.2 Для умного чанкинга

In [51]:
# Создаем документы для Qdrant с UUID
smart_docs = []
for i, chunk in enumerate(smart_chunks):
    smart_docs.append(Document(
        page_content=chunk.get('text', ''),
        metadata=chunk.get('metadata', '')
    ))

smart_uuids = [str(uuid4()) for _ in range(len(smart_docs))]
print(f"✅ Создано {len(smart_uuids)} 'умных' документов с UUID")

✅ Создано 398 'умных' документов с UUID


In [52]:
# Индексация в Qdrant
# Рекурсивная коллекция
vector_store_smart = QdrantVectorStore(
    embedding=embeddings_model,
    client=client,
    collection_name="sp60_smart",
)

#загрузка документов батчами
batch_size = 256
total_batches = (len(smart_docs) + batch_size - 1) // batch_size

with tqdm(total=total_batches, desc="Добавление батчей в Qdrant") as pbar:
    for i in range(0, len(smart_docs), batch_size):
        batch_docs = smart_docs[i:i+batch_size]
        batch_ids = smart_uuids[i:i+batch_size]
        vector_store_smart.add_documents(documents=batch_docs, ids=batch_ids)
        pbar.update(1)

print(f"✅ 'Умные' чанки ({len(smart_docs)}) проиндексированы и добавлены в Qdrant")

Добавление батчей в Qdrant: 100%|██████████| 2/2 [00:32<00:00, 16.47s/it]

✅ 'Умные' чанки (398) проиндексированы и добавлены в Qdrant


### 4. Тестирование поиска

#### 4.1 Простой семантический поиск в Qdrant

In [61]:
def get_results(query, k=5):
  recursive_results = vector_store_recursive.similarity_search_with_score(query, k=k)
  smart_results = vector_store_smart.similarity_search_with_score(query, k=k)

  print(f"✏ Рекурсивный чанкинг")
  for i, (doc, score) in enumerate(recursive_results):
    print(f"\n--- Результат {i+1} ---")
    # doc_id = doc.metadata.get('_id', 'Нет ID')
    # print(f"🆔 ID в базе: {doc_id}")
    print(f"📊 Similarity Score: {score:.4f}")
    print(f"Текст: {doc.page_content[:400]}...")

  print()
  print("=" * 50)

  print(f"🧠 Умный чанкинг")
  for i, (doc, score) in enumerate(smart_results):
    print(f"\n--- Результат {i+1} ---")
    # doc_id = doc.metadata.get('_id', 'Нет ID')
    # print(f"🆔 ID в базе: {doc_id}")
    print(f"📊 Similarity Score: {score:.4f}")
    print(f"Раздел: {doc.metadata.get('h2', '-')}")
    print(f"Текст: {doc.page_content[:400]}...")

In [63]:
query = "какова Максимальная масса хладагента?"
get_results(query, 3)

✏ Рекурсивный чанкинг

--- Результат 1 ---
📊 Similarity Score: 0.8471
Текст: $$$$
G _ { \max } = \Pi \Pi H \Psi \cdot L _ { o 0 u } ,
$$
<!-- snapshot: images/sp_60_formula_p80_2_snapshot.png -->$$

где L общ = V пом + L /4;

V пом - объем помещения, м 3 ;

L - подача наружного воздуха системой механической вентиляции, м 3 /ч.

В  помещениях,  масса  хладагента  при  аварийном  выбросе  в  которых  может превышать  ППНЧ  либо  10  %  НКПРП  следует  устанавливать  датчики ...

--- Результат 2 ---
📊 Similarity Score: 0.8357
Текст: 8.8 Максимальную и минимальную температуру и качество воды (незамерзающего раствора), подаваемой в испарительные и конденсаторные контуры холодильных машин, следует принимать в соответствии с техническими условиями на холодильные машины.

Расчетный перепад температур холодной и оборотной воды (раствора) в испарителе и конденсаторе рекомендуется принимать в пределах 4 °C - 6 °C.

Потери  холода  в ...

--- Результат 3 ---
📊 Similarity Score: 0.8236
Текст: Групп

#### 4.2 RAG

In [ ]:
def format_docs(docs, max_chars=None):
    """Форматирует документы для передачи в промпт.
    
    Args:
        docs: список документов
        max_chars: максимальная длина текста (None — без обрезки)
    """
    formatted = []

    for i, doc in enumerate(docs, 1):
        text = doc.page_content if max_chars is None else doc.page_content[:max_chars]
        doc_info = f"""
=== Документ {i} ===
Раздел: {doc.metadata.get('h1', '-')}
Подраздел: {doc.metadata.get('h2', '-')}
Текст: {text}
"""
        formatted.append(doc_info)

    return "\n".join(formatted)

print("✅ Функция форматирования создана")